# DeepLabV3+, SegFormer and CMX: seed 42

ImageNet-pretrained baselines. Maximum 70 epochs. Internet access required for initial dependency/weight/repository retrieval.

Source: `gd-second.ipynb`, zero-based cell 15.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
# ============================================================
# GD-RGBMS-DB | STRONG BASELINES | KAGGLE ONE-CELL SCRIPT
# Models:
#   1) DeepLabV3+ + ImageNet-pretrained ResNet-50 (6-ch early fusion)
#   2) SegFormer + ImageNet-pretrained MiT-B1 (6-ch early fusion)
#   3) CMX + MiT-B0 (official CMX architecture adapted to RGB + 3-band MS)
#
# Experiment policy:
#   - EXACTLY 3 training runs total: one run per model
#   - Seed 42 only for this first-pass comparison
#   - Fixed initial learning rate = 1e-4 for all three models
#   - Official split: 280 / 60 / 60
#   - Input: RGB + Green + Red-Edge + NIR = 6 channels
#   - Same train-only normalization, augmentation, loss, optimizer family,
#     scheduler, early stopping, validation-only threshold selection, no TTA
#   - No learning-rate sweep and no seeds 123/2026 in this script
#
# IMPORTANT:
#   Kaggle -> Settings -> Internet = ON (needed for pip packages, ImageNet
#   weights, and the official CMX repository on first run).
# ============================================================

from __future__ import annotations

import os
import sys
import csv
import gc
import json
import time
import math
import random
import shutil
import subprocess
import importlib
from pathlib import Path
from types import SimpleNamespace

# ------------------------------------------------------------
# 0. INSTALL / IMPORT EXTERNAL DEPENDENCIES
# ------------------------------------------------------------

def _pip_install_if_needed():
    try:
        import segmentation_models_pytorch  # noqa: F401
        import timm  # noqa: F401
        return
    except Exception:
        pass
    print("Installing segmentation-models-pytorch + dependencies ...")
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "segmentation-models-pytorch==0.5.0",
        "timm>=1.0.15",
        "huggingface_hub>=0.25.0",
        "safetensors>=0.4.0",
    ]
    try:
        subprocess.check_call(cmd)
    except Exception as e:
        raise RuntimeError(
            "Package installation failed. In Kaggle, turn Internet ON and rerun the cell."
        ) from e

_pip_install_if_needed()

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Dataset path from your Kaggle runs; auto-discovery is used if it changes.
PREFERRED_DATASET_DIR = Path("/kaggle/input/datasets/hemal7/gd-rgbms-db")

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
MASTER_OUT_DIR = WORK_DIR / "gd_strong_baselines_seed42_3models"
MASTER_OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = (896, 672)  # PIL = (width, height)
RUN_SEED = 42
FIXED_LR = 1e-4

MAX_EPOCHS = 70
EARLY_STOPPING_PATIENCE = 12
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
CLIP_GRAD_NORM = 1.0

# Per-model physical batch can differ only for GPU-memory reasons.
# Effective batch size remains 4 for all models.
MODEL_SPECS = {
    "deeplabv3plus_resnet50": {
        "label": "DeepLabV3+ ResNet-50",
        "batch_size": 2,
        "accumulation_steps": 2,
        "pretraining": "ImageNet",
        "input_strategy": "6-channel early fusion",
    },
    "segformer_mit_b1": {
        "label": "SegFormer MiT-B1",
        "batch_size": 1,
        "accumulation_steps": 4,
        "pretraining": "ImageNet",
        "input_strategy": "6-channel early fusion",
    },
    "cmx_mit_b0": {
        "label": "CMX MiT-B0",
        "batch_size": 1,
        "accumulation_steps": 4,
        "pretraining": "ImageNet MiT-B0 copied to both CMX branches",
        "input_strategy": "RGB branch + MS(G,RE,NIR) branch",
    },
}

RUN_MODELS = [
    "deeplabv3plus_resnet50",
    "segformer_mit_b1",
    "cmx_mit_b0",
]

CACHE_TRAIN_AND_VALIDATION = True
SHOW_PROGRESS = False

MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)
ALL_CHANNEL_NAMES = ["R", "G", "B", "MS_Green", "Red-Edge", "NIR"]
ALL_MEAN = np.array(
    [
        0.41819795966148376,
        0.46573567390441895,
        0.2516770660877228,
        0.21822915971279144,
        0.16206900775432587,
        0.23585167527198792,
    ],
    dtype=np.float32,
)
ALL_STD = np.array(
    [
        0.1540451943874359,
        0.1569177359342575,
        0.13375557959079742,
        0.07147330790758133,
        0.05124884843826294,
        0.04640943184494972,
    ],
    dtype=np.float32,
)

TRAIN_THRESHOLDS = np.round(np.arange(0.20, 0.851, 0.05), 2).tolist()
FINAL_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

if DEVICE.type != "cuda":
    print("WARNING: CUDA not detected. These models are intended to run on a Kaggle GPU.")


def pbar(*args, **kwargs):
    kwargs["disable"] = not SHOW_PROGRESS
    return tqdm(*args, **kwargs)


# ============================================================
# 2. DATASET DISCOVERY / REPRODUCIBILITY
# ============================================================

def read_list(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def valid_root(path: Path):
    required = [
        path / "RGB",
        path / "MS",
        path / "Mark",
        path / "train_list.txt",
        path / "val_list.txt",
        path / "test_list.txt",
    ]
    if not all(p.exists() for p in required):
        return False
    try:
        return (
            len(read_list(path / "train_list.txt")) == 280
            and len(read_list(path / "val_list.txt")) == 60
            and len(read_list(path / "test_list.txt")) == 60
        )
    except Exception:
        return False


def find_dataset_root():
    candidates = [
        PREFERRED_DATASET_DIR,
        Path("/kaggle/input/gd-rgbms-db"),
        Path("/kaggle/input/gd-rgbms-db/GD-RGBMS-DB"),
    ]
    for p in candidates:
        if valid_root(p):
            return p

    root = Path("/kaggle/input")
    if root.exists():
        for train_file in root.rglob("train_list.txt"):
            p = train_file.parent
            if valid_root(p):
                return p

    raise FileNotFoundError(
        "Could not find GD-RGBMS-DB. Dataset root must contain RGB/, MS/, Mark/, "
        "train_list.txt, val_list.txt, test_list.txt."
    )


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


DATASET_DIR = find_dataset_root()
RGB_DIR = DATASET_DIR / "RGB"
MS_DIR = DATASET_DIR / "MS"
MARK_DIR = DATASET_DIR / "Mark"

TRAIN_NAMES = read_list(DATASET_DIR / "train_list.txt")
VAL_NAMES = read_list(DATASET_DIR / "val_list.txt")
TEST_NAMES = read_list(DATASET_DIR / "test_list.txt")

assert len(TRAIN_NAMES) == 280
assert len(VAL_NAMES) == 60
assert len(TEST_NAMES) == 60
assert not (set(TRAIN_NAMES) & set(VAL_NAMES))
assert not (set(TRAIN_NAMES) & set(TEST_NAMES))
assert not (set(VAL_NAMES) & set(TEST_NAMES))

print("=" * 110)
print("KAGGLE | GD-RGBMS-DB | STRONG BASELINES | 3 MODELS | SEED 42 ONLY")
print("=" * 110)
print("Dataset      :", DATASET_DIR)
print("Device       :", DEVICE)
print("PyTorch      :", torch.__version__)
print("SMP          :", smp.__version__)
print("Split        :", len(TRAIN_NAMES), len(VAL_NAMES), len(TEST_NAMES))
print("Input size   :", TARGET_SIZE)
print("Models       :", [MODEL_SPECS[m]["label"] for m in RUN_MODELS])
print("Seed         :", RUN_SEED)
print("Initial LR   :", FIXED_LR, "(same for all three models; no LR sweep)")
print("Total runs   :", len(RUN_MODELS))
print("=" * 110)


# ============================================================
# 3. RAW DATA / CACHE
# ============================================================

def clean_stem(name: str):
    return Path(str(name).strip()).stem


def find_path(directory: Path, name_or_stem: str, extensions):
    raw_name = Path(str(name_or_stem).strip()).name

    exact = directory / raw_name
    if exact.exists() and exact.is_file():
        return exact

    stem = Path(raw_name).stem
    for ext in extensions:
        p = directory / f"{stem}{ext}"
        if p.exists() and p.is_file():
            return p

    matches = [p for p in directory.glob(f"{stem}.*") if p.is_file()]
    if matches:
        matches.sort()
        return matches[0]

    raise FileNotFoundError(
        f"Could not find paired file for list entry {name_or_stem!r} "
        f"(stem={stem!r}) in {directory}"
    )


def mask_to_binary(image: Image.Image):
    rgb = np.asarray(image.convert("RGB"), dtype=np.int16)
    return (np.abs(rgb - MASK_POSITIVE_RGB).sum(axis=2) < 80).astype(np.float32)


def load_raw(name: str):
    stem = clean_stem(name)
    rgb_path = find_path(
        RGB_DIR, name, [".JPG", ".jpg", ".JPEG", ".jpeg", ".PNG", ".png"]
    )
    ms_path = find_path(
        MS_DIR, stem, [".PNG", ".png", ".JPG", ".jpg", ".JPEG", ".jpeg"]
    )
    mask_path = find_path(
        MARK_DIR, stem, [".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG"]
    )

    with Image.open(rgb_path) as im:
        rgb = np.asarray(
            im.convert("RGB").resize(TARGET_SIZE, BILINEAR), dtype=np.float32
        ) / 255.0
    with Image.open(ms_path) as im:
        ms = np.asarray(
            im.convert("RGB").resize(TARGET_SIZE, BILINEAR), dtype=np.float32
        ) / 255.0
    with Image.open(mask_path) as im:
        mask = mask_to_binary(im.convert("RGB").resize(TARGET_SIZE, NEAREST))

    image = np.concatenate([rgb, ms], axis=2).astype(np.float32)
    return image, mask, stem


RAW_CACHE = {}
if CACHE_TRAIN_AND_VALIDATION:
    print("Preloading train + validation images to RAM ...")
    preload_names = sorted(set(TRAIN_NAMES + VAL_NAMES))
    for name in pbar(preload_names, desc="Preloading"):
        image, mask, stem = load_raw(name)
        RAW_CACHE[name] = (image.astype(np.float16), mask.astype(np.uint8), stem)
    print(f"Cached {len(RAW_CACHE)} images.")


def get_raw(name: str):
    cached = RAW_CACHE.get(name)
    if cached is not None:
        image, mask, stem = cached
        return image.astype(np.float32), mask.astype(np.float32), stem
    return load_raw(name)


# ============================================================
# 4. DATASET / LOADERS
# ============================================================

class SixChannelDataset(Dataset):
    def __init__(self, names, augment=False):
        self.names = list(names)
        self.augment = augment
        self.mean = ALL_MEAN.reshape(1, 1, 6)
        self.std = ALL_STD.reshape(1, 1, 6)

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image, mask, stem = get_raw(name)

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()
            if random.random() < 0.5:
                image = np.flip(image, axis=0).copy()
                mask = np.flip(mask, axis=0).copy()
            if random.random() < 0.75:
                gain = np.random.uniform(0.88, 1.12, size=(1, 1, 3)).astype(np.float32)
                bias = np.random.uniform(-0.035, 0.035, size=(1, 1, 3)).astype(np.float32)
                image[..., :3] = np.clip(image[..., :3] * gain + bias, 0.0, 1.0)

        image = ((image - self.mean) / self.std).astype(np.float32)
        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask[None]).float()
        return image, mask, stem


def make_loaders(seed: int, batch_size: int):
    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        SixChannelDataset(TRAIN_NAMES, augment=True),
        batch_size=batch_size,
        shuffle=True,
        generator=g,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    val_loader = DataLoader(
        SixChannelDataset(VAL_NAMES, augment=False),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    test_loader = DataLoader(
        SixChannelDataset(TEST_NAMES, augment=False),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    return train_loader, val_loader, test_loader


# ============================================================
# 5. MODEL DEFINITIONS
# ============================================================

class EnsureFullResolution(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        y = self.model(x)
        if isinstance(y, (tuple, list)):
            y = y[0]
        if y.shape[-2:] != x.shape[-2:]:
            y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return y


def build_deeplabv3plus_resnet50():
    model = smp.DeepLabV3Plus(
        encoder_name="resnet50",
        encoder_weights="imagenet",
        encoder_output_stride=16,
        decoder_channels=256,
        decoder_atrous_rates=(12, 24, 36),
        in_channels=6,
        classes=1,
        activation=None,
    )
    return EnsureFullResolution(model)


def build_segformer_mit_b1():
    model = smp.Segformer(
        encoder_name="mit_b1",
        encoder_weights="imagenet",
        in_channels=6,
        classes=1,
        activation=None,
        decoder_segmentation_channels=256,
    )
    return EnsureFullResolution(model)


CMX_REPO_URL = "https://github.com/huaaaliu/RGBX_Semantic_Segmentation.git"
CMX_REPO_DIR = WORK_DIR / "RGBX_Semantic_Segmentation_official"
_CMX_CLASS_CACHE = None
_CMX_REPO_COMMIT = None


def ensure_official_cmx_repo():
    global _CMX_REPO_COMMIT
    if not (CMX_REPO_DIR / "models" / "builder.py").exists():
        if CMX_REPO_DIR.exists():
            shutil.rmtree(CMX_REPO_DIR)
        print("Cloning official CMX repository ...")
        try:
            subprocess.check_call(
                ["git", "clone", "--depth", "1", CMX_REPO_URL, str(CMX_REPO_DIR)]
            )
        except Exception as e:
            raise RuntimeError(
                "Could not clone the official CMX repository. Turn Kaggle Internet ON and rerun."
            ) from e

    try:
        _CMX_REPO_COMMIT = subprocess.check_output(
            ["git", "-C", str(CMX_REPO_DIR), "rev-parse", "HEAD"], text=True
        ).strip()
    except Exception:
        _CMX_REPO_COMMIT = "unknown"

    if str(CMX_REPO_DIR) not in sys.path:
        sys.path.insert(0, str(CMX_REPO_DIR))


def _load_official_cmx_class():
    global _CMX_CLASS_CACHE
    if _CMX_CLASS_CACHE is not None:
        return _CMX_CLASS_CACHE

    ensure_official_cmx_repo()

    # Avoid stale generic modules with the same top-level names used by CMX.
    for prefix in ("models", "utils", "engine"):
        if prefix in sys.modules:
            mod_file = getattr(sys.modules[prefix], "__file__", "") or ""
            if str(CMX_REPO_DIR) not in mod_file:
                for key in list(sys.modules.keys()):
                    if key == prefix or key.startswith(prefix + "."):
                        del sys.modules[key]

    builder = importlib.import_module("models.builder")
    _CMX_CLASS_CACHE = builder.EncoderDecoder
    return _CMX_CLASS_CACHE


def _initialize_cmx_decoder(model):
    # The official builder normally applies Kaiming initialization through its
    # training framework. We instantiate with criterion=None to avoid requiring
    # an external local pretrained checkpoint, so initialize the decoder here.
    for module in model.decode_head.modules():
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)


def _copy_imagenet_mit_b0_to_cmx_dual_backbone(cmx_backbone):
    # SMP's MiT-B0 is a port of the official SegFormer encoder and exposes the
    # same patch_embed / block / norm naming. Copy the ImageNet state to both
    # RGB and X branches of the official CMX dual-path backbone.
    source = smp.encoders.get_encoder(
        "mit_b0", in_channels=3, depth=5, weights="imagenet"
    )
    src = source.state_dict()
    target = cmx_backbone.state_dict()

    mapped = {}
    for key, value in src.items():
        if not key.startswith(("patch_embed", "block", "norm")):
            continue
        if key in target and target[key].shape == value.shape:
            mapped[key] = value

        if key.startswith("patch_embed"):
            other = key.replace("patch_embed", "extra_patch_embed", 1)
        elif key.startswith("block"):
            other = key.replace("block", "extra_block", 1)
        elif key.startswith("norm"):
            other = key.replace("norm", "extra_norm", 1)
        else:
            other = None

        if other is not None and other in target and target[other].shape == value.shape:
            mapped[other] = value.clone()

    if len(mapped) < 50:
        raise RuntimeError(
            f"CMX ImageNet initialization mapping looks incomplete: only {len(mapped)} tensors matched."
        )

    info = cmx_backbone.load_state_dict(mapped, strict=False)
    print(
        f"CMX pretrained initialization: loaded {len(mapped)} tensors into dual MiT-B0 backbone "
        f"(missing after partial load={len(info.missing_keys)}, unexpected={len(info.unexpected_keys)})."
    )
    del source, src, target, mapped
    gc.collect()


class CMXBinaryWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        EncoderDecoder = _load_official_cmx_class()

        cfg = SimpleNamespace(
            backbone="mit_b0",
            decoder="MLPDecoder",
            num_classes=1,
            decoder_embed_dim=256,
            pretrained_model=None,
            bn_eps=1e-5,
            bn_momentum=0.1,
        )

        # criterion=None prevents the official builder from requiring a local
        # pretrained checkpoint path. We then load ImageNet MiT-B0 weights into
        # both branches explicitly below.
        self.model = EncoderDecoder(cfg=cfg, criterion=None, norm_layer=nn.BatchNorm2d)
        _initialize_cmx_decoder(self.model)
        _copy_imagenet_mit_b0_to_cmx_dual_backbone(self.model.backbone)

    def forward(self, x):
        rgb = x[:, :3]
        ms = x[:, 3:6]
        y = self.model(rgb, ms)
        if isinstance(y, (tuple, list)):
            y = y[0]
        if y.shape[-2:] != x.shape[-2:]:
            y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return y


def build_model(model_key: str):
    if model_key == "deeplabv3plus_resnet50":
        model = build_deeplabv3plus_resnet50()
    elif model_key == "segformer_mit_b1":
        model = build_segformer_mit_b1()
    elif model_key == "cmx_mit_b0":
        model = CMXBinaryWrapper()
    else:
        raise KeyError(model_key)

    model = model.to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: total={total:,} | trainable={trainable:,}")
    return model, total, trainable


# ============================================================
# 6. LOSS / METRICS
# ============================================================

class ComboLoss(nn.Module):
    def forward(self, logits, targets):
        targets = targets.float()
        bce_map = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        bce = bce_map.mean()

        probs = torch.sigmoid(logits)
        pt = probs * targets + (1.0 - probs) * (1.0 - targets)
        focal = (bce_map * ((1.0 - pt) ** 2.0)).mean()

        dims = tuple(range(1, logits.ndim))
        intersection = (probs * targets).sum(dims)
        denominator = probs.sum(dims) + targets.sum(dims)
        dice_loss = 1.0 - (
            (2.0 * intersection + 1e-6) / (denominator + 1e-6)
        ).mean()

        return 0.35 * bce + 0.25 * focal + 0.40 * dice_loss


criterion = ComboLoss()


def confusion_numpy(probability, mask, threshold):
    pred = probability >= threshold
    target = mask >= 0.5
    return [
        int(np.logical_and(pred, target).sum()),
        int(np.logical_and(pred, ~target).sum()),
        int(np.logical_and(~pred, target).sum()),
        int(np.logical_and(~pred, ~target).sum()),
    ]


def confusion_tensor(probability, mask, threshold):
    pred = probability >= threshold
    target = mask >= 0.5
    return [
        int((pred & target).sum().item()),
        int((pred & ~target).sum().item()),
        int((~pred & target).sum().item()),
        int((~pred & ~target).sum().item()),
    ]


def add_confusion(a, b):
    return [int(x + y) for x, y in zip(a, b)]


def metric_dict(values):
    tp, fp, fn, tn = values
    e = 1e-7
    return {
        "dice": (2 * tp + e) / (2 * tp + fp + fn + e),
        "iou": (tp + e) / (tp + fp + fn + e),
        "precision": (tp + e) / (tp + fp + e),
        "recall": (tp + e) / (tp + fn + e),
        "accuracy": (tp + tn + e) / (tp + fp + fn + tn + e),
    }


def save_csv(path: Path, rows):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# 7. EVALUATION HELPERS
# ============================================================

@torch.no_grad()
def collect_records(loader, model, desc):
    model.eval()
    records = []
    for images, masks, stems in pbar(loader, desc=desc, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            logits = model(images)
            probs = torch.sigmoid(logits)

        records.append(
            {
                "stem": stems[0],
                "probability": probs[0, 0].float().cpu().numpy(),
                "mask": masks[0, 0].float().numpy(),
            }
        )
    return records


def search_threshold(records, thresholds):
    rows = []
    for threshold in thresholds:
        total = [0, 0, 0, 0]
        for record in records:
            total = add_confusion(
                total,
                confusion_numpy(record["probability"], record["mask"], threshold),
            )
        row = metric_dict(total)
        row["threshold"] = float(threshold)
        rows.append(row)
    best = max(rows, key=lambda row: row["dice"])
    return best, rows


# ============================================================
# 8. TRAIN ONE CANDIDATE
# ============================================================

def train_candidate(model_key: str, seed: int, learning_rate: float, candidate_tag: str):
    spec = MODEL_SPECS[model_key]
    batch_size = spec["batch_size"]
    accumulation_steps = spec["accumulation_steps"]

    run_dir = MASTER_OUT_DIR / model_key / candidate_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / "best_checkpoint.pt"

    print("\n" + "#" * 110)
    print(
        f"TRAIN | {spec['label']} | seed={seed} | lr={learning_rate:.1e} | "
        f"batch={batch_size} | accum={accumulation_steps} | effective batch={batch_size*accumulation_steps}"
    )
    print("#" * 110)

    seed_everything(seed)
    train_loader, val_loader, _ = make_loaders(seed, batch_size)
    model, total_params, trainable_params = build_model(model_key)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

    history_rows = []
    best_val_dice = -1.0
    best_epoch = 0
    best_coarse_threshold = None

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss_sum = 0.0

        for step, (images, masks, _) in enumerate(
            pbar(train_loader, desc=f"{model_key} s{seed} train {epoch}", leave=False),
            start=1,
        ):
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
                scaled_loss = loss / accumulation_steps

            scaler.scale(scaled_loss).backward()

            if step % accumulation_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            loss_sum += float(loss.item())

        train_loss = loss_sum / max(len(train_loader), 1)

        # Validation with coarse threshold grid for checkpoint selection.
        model.eval()
        threshold_totals = {th: [0, 0, 0, 0] for th in TRAIN_THRESHOLDS}
        val_loss_sum = 0.0
        with torch.no_grad():
            for images, masks, _ in pbar(
                val_loader, desc=f"{model_key} s{seed} val {epoch}", leave=False
            ):
                images = images.to(DEVICE, non_blocking=True)
                masks = masks.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                    logits = model(images)
                    val_loss = criterion(logits, masks)
                    probs = torch.sigmoid(logits)
                val_loss_sum += float(val_loss.item())
                for th in TRAIN_THRESHOLDS:
                    threshold_totals[th] = add_confusion(
                        threshold_totals[th],
                        confusion_tensor(probs, masks, th),
                    )

        val_rows = []
        for th in TRAIN_THRESHOLDS:
            row = metric_dict(threshold_totals[th])
            row["threshold"] = float(th)
            val_rows.append(row)
        val_best = max(val_rows, key=lambda r: r["dice"])
        val_best["loss"] = val_loss_sum / max(len(val_loader), 1)

        current_lr = float(optimizer.param_groups[0]["lr"])
        scheduler.step(val_best["dice"])

        hist = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_best["loss"],
            "val_dice": val_best["dice"],
            "val_iou": val_best["iou"],
            "val_precision": val_best["precision"],
            "val_recall": val_best["recall"],
            "val_accuracy": val_best["accuracy"],
            "val_threshold": val_best["threshold"],
            "learning_rate": current_lr,
            "minutes": (time.time() - epoch_start) / 60.0,
        }
        history_rows.append(hist)

        print(
            f"{model_key} | seed {seed} | lr0 {learning_rate:.1e} | epoch {epoch:03d} | "
            f"loss {train_loss:.4f} | val Dice {val_best['dice']:.4f} | "
            f"IoU {val_best['iou']:.4f} | P {val_best['precision']:.4f} | "
            f"R {val_best['recall']:.4f} | th {val_best['threshold']:.2f} | "
            f"lr {current_lr:.2e}"
        )

        if val_best["dice"] > best_val_dice:
            best_val_dice = float(val_best["dice"])
            best_epoch = epoch
            best_coarse_threshold = float(val_best["threshold"])
            torch.save(
                {
                    "model": model.state_dict(),
                    "model_key": model_key,
                    "seed": seed,
                    "initial_learning_rate": learning_rate,
                    "best_epoch": best_epoch,
                    "best_validation_dice": best_val_dice,
                    "coarse_validation_threshold": best_coarse_threshold,
                    "total_params": total_params,
                    "trainable_params": trainable_params,
                },
                checkpoint_path,
            )

        if epoch - best_epoch >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping after {EARLY_STOPPING_PATIENCE} non-improving epochs."
            )
            break

    save_csv(run_dir / "training_history.csv", history_rows)

    meta = {
        "model_key": model_key,
        "model_label": spec["label"],
        "seed": seed,
        "initial_learning_rate": learning_rate,
        "best_epoch": best_epoch,
        "best_validation_dice": best_val_dice,
        "coarse_validation_threshold": best_coarse_threshold,
        "checkpoint_path": str(checkpoint_path),
        "run_dir": str(run_dir),
        "total_params": total_params,
        "trainable_params": trainable_params,
        "batch_size": batch_size,
        "accumulation_steps": accumulation_steps,
    }

    with open(run_dir / "candidate_summary.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return meta


# ============================================================
# 9. FINAL VALIDATION THRESHOLD + TEST FOR A SELECTED CHECKPOINT
# ============================================================

def evaluate_selected_checkpoint(meta):
    model_key = meta["model_key"]
    seed = int(meta["seed"])
    spec = MODEL_SPECS[model_key]
    run_dir = Path(meta["run_dir"])

    print("\n" + "-" * 110)
    print(
        f"FINAL EVAL | {spec['label']} | seed={seed} | selected lr={meta['initial_learning_rate']:.1e}"
    )
    print("-" * 110)

    seed_everything(seed)
    _, val_loader, test_loader = make_loaders(seed, batch_size=1)
    model, total_params, trainable_params = build_model(model_key)

    checkpoint = torch.load(meta["checkpoint_path"], map_location=DEVICE)
    model.load_state_dict(checkpoint["model"])

    val_records = collect_records(
        val_loader, model, f"{model_key} seed {seed} fine validation threshold"
    )
    val_best, threshold_rows = search_threshold(val_records, FINAL_THRESHOLDS)
    selected_threshold = float(val_best["threshold"])
    save_csv(run_dir / "validation_threshold_search.csv", threshold_rows)

    test_records = collect_records(
        test_loader, model, f"{model_key} seed {seed} final no-TTA test"
    )

    total = [0, 0, 0, 0]
    per_image_rows = []
    for record in test_records:
        values = confusion_numpy(
            record["probability"], record["mask"], selected_threshold
        )
        total = add_confusion(total, values)
        row = metric_dict(values)
        row.update(
            {
                "stem": record["stem"],
                "threshold": selected_threshold,
                "tp": values[0],
                "fp": values[1],
                "fn": values[2],
                "tn": values[3],
            }
        )
        per_image_rows.append(row)

    result = metric_dict(total)
    result.update(
        {
            "model_key": model_key,
            "model_label": spec["label"],
            "seed": seed,
            "initial_learning_rate": float(meta["initial_learning_rate"]),
            "best_epoch": int(checkpoint["best_epoch"]),
            "best_validation_dice": float(checkpoint["best_validation_dice"]),
            "fine_validation_dice": float(val_best["dice"]),
            "selected_threshold": selected_threshold,
            "total_params": int(total_params),
            "trainable_params": int(trainable_params),
            "pretraining": spec["pretraining"],
            "input_strategy": spec["input_strategy"],
            "tp": total[0],
            "fp": total[1],
            "fn": total[2],
            "tn": total[3],
        }
    )

    save_csv(run_dir / "final_test_metrics.csv", [result])
    save_csv(run_dir / "per_image_test_metrics.csv", per_image_rows)

    print(
        f"RESULT | {spec['label']} | seed {seed} | "
        f"Dice={result['dice']:.4f} | IoU={result['iou']:.4f} | "
        f"P={result['precision']:.4f} | R={result['recall']:.4f} | "
        f"Acc={result['accuracy']:.4f} | th={selected_threshold:.2f} | "
        f"best epoch={result['best_epoch']}"
    )

    del model, val_loader, test_loader, val_records, test_records
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result


# ============================================================
# 10. FIRST-PASS EXPERIMENT: EXACTLY ONE RUN PER MODEL
# ============================================================

def run_model_experiment(model_key: str):
    spec = MODEL_SPECS[model_key]
    model_dir = MASTER_OUT_DIR / model_key
    model_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 110)
    print(f"SINGLE RUN: {spec['label']} | seed {RUN_SEED} | initial LR {FIXED_LR:.1e}")
    print("=" * 110)

    tag = f"seed{RUN_SEED}_lr{FIXED_LR:.0e}".replace("-", "m")
    meta = train_candidate(model_key, RUN_SEED, FIXED_LR, tag)
    result = evaluate_selected_checkpoint(meta)

    save_csv(model_dir / "seed42_result.csv", [result])
    return result


# ============================================================
# 11. RUN THE THREE MODELS = EXACTLY THREE TRAINING RUNS
# ============================================================

all_results = []
for model_key in RUN_MODELS:
    all_results.append(run_model_experiment(model_key))

save_csv(MASTER_OUT_DIR / "all_models_seed42_results.csv", all_results)

# Save experiment-level configuration / provenance.
experiment_info = {
    "dataset": str(DATASET_DIR),
    "split_counts": {
        "train": len(TRAIN_NAMES),
        "validation": len(VAL_NAMES),
        "test": len(TEST_NAMES),
    },
    "target_size_pil_wh": list(TARGET_SIZE),
    "channels": ALL_CHANNEL_NAMES,
    "normalization_mean": ALL_MEAN.tolist(),
    "normalization_std": ALL_STD.tolist(),
    "augmentation": "H/V flips + RGB-only gain/bias",
    "loss": "0.35 BCE + 0.25 focal(gamma=2) + 0.40 soft Dice",
    "optimizer": "AdamW",
    "weight_decay": WEIGHT_DECAY,
    "initial_learning_rate": FIXED_LR,
    "scheduler": "ReduceLROnPlateau(mode=max,factor=0.5,patience=4,min_lr=1e-6)",
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "threshold_search": "validation only: coarse 0.20-0.85 step 0.05 during training; final 0.20-0.80 step 0.01",
    "test_time_augmentation": "none",
    "seed": RUN_SEED,
    "total_training_runs": len(RUN_MODELS),
    "model_specs": MODEL_SPECS,
    "segmentation_models_pytorch_version": smp.__version__,
    "pytorch_version": torch.__version__,
    "official_cmx_repository": CMX_REPO_URL,
    "official_cmx_commit": _CMX_REPO_COMMIT,
}
with open(MASTER_OUT_DIR / "experiment_configuration.json", "w", encoding="utf-8") as f:
    json.dump(experiment_info, f, indent=2)

# Create a single ZIP with histories, checkpoints and per-image metrics.
zip_base = WORK_DIR / "gd_strong_baselines_seed42_3models_complete"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=MASTER_OUT_DIR))

print("\n" + "=" * 110)
print("ALL THREE SINGLE-SEED BASELINE RUNS FINISHED")
print("=" * 110)
for r in all_results:
    print(
        f"{r['model_label']:<28} | "
        f"Dice {r['dice']:.4f} | IoU {r['iou']:.4f} | "
        f"P {r['precision']:.4f} | R {r['recall']:.4f} | "
        f"Acc {r['accuracy']:.4f} | th {r['selected_threshold']:.2f} | "
        f"epoch {r['best_epoch']}"
    )
print("\nMaster output:", MASTER_OUT_DIR)
print("ZIP          :", zip_path)
